# Tutorial 4.2: The Adjoint Method and Topology Optimisation

In this tutorial, we introduce `firedrake.adjoint` to automatically compute exact analytical gradients. 

We will perform a computational scaling analysis to definitively show why Finite Differences (as seen in Tutorial 4.1) are practically impossible for large meshes. Following that, we will execute an optimisation using Adjoint sensitivities.

## 1. Standard Imports and Setup

First, we import Firedrake, the `adjoint` module, and our optimisation and visualisation libraries. MMA can be imported from the GitHub repository.

In [ ]:
try:
    from firedrake import *
    from firedrake.adjoint import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *
    from firedrake.adjoint import *

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML, display
import time
from tqdm.auto import tqdm

try:
    from mma import gcmmasub, asymp, concheck, raaupdate
except ImportError:
    print("Error: `mma.py` not found. Please upload it to your workspace.")

## 2. The Adjoint Method

In our previous tutorial, computing gradients via Finite Differences required a full PDE solve for **every single design variable**. For a mesh with thousands of nodes, this implies thousands of PDE solves per optimisation step: a cost that is prohibitive.

### The Adjoint Approach
Let our objective function be $J(u, p)$, where $u$ is the state variable (e.g., Temperature) and $p$ is our design parameter field (e.g., thickness). The system is governed by the PDE constraint $F(u, p) = 0$.

We construct the Lagrangian by introducing an adjoint variable (Lagrange multiplier) $\lambda$:
$$\mathcal{L}(u, p, \lambda) = J(u, p) + \lambda^T F(u, p)$$

To find the total sensitivity of the objective with respect to the design parameters, we take the total derivative:
$$\frac{d\mathcal{L}}{dp} = \frac{\partial J}{\partial p} + \frac{\partial J}{\partial u}\frac{du}{dp} + \lambda^T \left( \frac{\partial F}{\partial p} + \frac{\partial F}{\partial u}\frac{du}{dp} \right)$$

Rearranging to group the terms dependent on $\frac{du}{dp}$ (the bit that is expensive to compute):
$$\frac{d\mathcal{L}}{dp} = \frac{\partial J}{\partial p} + \lambda^T \frac{\partial F}{\partial p} + \left( \frac{\partial J}{\partial u} + \lambda^T \frac{\partial F}{\partial u} \right) \frac{du}{dp}$$

By intentionally choosing $\lambda$ such that the term in the brackets equals zero, we eliminate the need to compute $\frac{du}{dp}$. This yields the **Adjoint Equation**:
$$\left[\frac{\partial F}{\partial u}\right]^T \lambda = -\left[\frac{\partial J}{\partial u}\right]^T$$

Once $\lambda$ is found (requiring only **one** additional linear solve), the exact gradient becomes trivial to compute regardless of how many parameters $p$ exist:
$$\frac{dJ}{dp} = \frac{\partial J}{\partial p} + \lambda^T \frac{\partial F}{\partial p}$$

### How Firedrake Records the Adjoint

Before diving into the maths, it is helpful to understand how Firedrake actually computes these gradients behind the scenes. It uses an automated "taping" mechanism. 

*   **The Forward Pass:** As your simulation runs, Firedrake records every mathematically significant operation (such as `solve` or `assemble`) onto a directed acyclic graph known as the "tape".
*   **Symbolic Differentiation:** Because Firedrake is built on the Unified Form Language (UFL), it understands the underlying calculus. It differentiates the mathematical expression of the weak form itself, rather than using standard line-by-line automatic differentiation. 
*   **The Reverse Pass:** Once the forward simulation is complete and you define your objective functional, Firedrake "rewinds" the tape. It automatically derives and solves the corresponding adjoint equations, propagating sensitivities back to your original design parameters.

## 3. Optimization Formulation

To find the most efficient distribution of material, we define a formal optimization problem. Our goal is to minimize the thermal resistance of the plate (represented by the peak temperature) while staying within a strict "material budget."

### The Objective Function
We aim to minimize the maximum temperature $T$ across the domain $\Omega$. Because the $\max(\cdot)$ function is non-differentiable, we use a smooth **$p$-norm approximation** as our objective functional $J$:

$$J(T) = \left( \int_{\Omega} T^p \, dx \right)^{1/p}$$

In this tutorial, we set **$p = 4$**. This value is high enough to penalize local "hot spots" effectively while maintaining a functional that is smooth enough for the gradient-based optimizer to navigate.

### The Design Variable and Physical Mapping
The optimizer manipulates a continuous density field $\rho(x)$, bounded such that $0 \le \rho(x) \le 1$. To ensure numerical stability (avoiding a zero-conductivity matrix) and to represent physical reality, we map this density to a physical thickness field $t(\rho)$:

$$t(\rho) = t_{\text{min}} + \rho(t_{\text{max}} - t_{\text{min}})$$

* **$t_{\text{min}} = 0.01$**: The minimum thickness, ensuring the PDE remains well-posed.
* **$t_{\text{max}} = 0.5$**: The maximum allowable thickness at any single point.

### Constraints
The optimization is performed subject to two critical constraints:

1.  **The State Equation (PDE Constraint):** The temperature field $T$ must physically satisfy the steady-state heat equation for the current thickness distribution. In the weak form, we seek $T \in V$ such that:
    $$\int_{\Omega} k \, t(\rho) \nabla T \cdot \nabla v \, dx = \int_{\Omega} f_{\text{source}} v \, dx \quad \forall v \in V$$

2.  **The Volume Constraint:** The total integrated volume of the material must not exceed our target budget, $V_{\text{target}}$ (which we have set to **12%** of the domain's maximum capacity):
    $$\int_{\Omega} t(\rho) \, dx \le V_{\text{target}}$$

In [ ]:
Lx, Ly = 1.0, 1.0
mesh = RectangleMesh(100, 100, Lx, Ly)

mass_limit = 0.12 # Target total volume
V = FunctionSpace(mesh, "CG", 1)
t_min_adj, t_max_adj = 0.01, 0.5
rho = Function(V, name="DesignVariable").assign((mass_limit - t_min_adj) / (t_max_adj - t_min_adj))
t_phys = t_min_adj + rho * (t_max_adj - t_min_adj)

T = Function(V, name="Temperature")
T_trial, v_test = TrialFunction(V), TestFunction(V)
x, y = SpatialCoordinate(mesh)

k = Constant(15.0)
f_source = 25000.0 * exp(-15.0 * ((x - Lx/2)**2 + (y - Ly/2)**2))
t_cond = t_min_adj + rho**1 * (t_max_adj - t_min_adj)

thermal_form = k * t_cond * inner(grad(T_trial), grad(v_test)) * dx - f_source * v_test * dx
a_form, L_form = lhs(thermal_form), rhs(thermal_form)
bc = DirichletBC(V, Constant(20.0), "on_boundary")

problem = LinearVariationalProblem(a_form, L_form, T, bcs=bc)
solver = LinearVariationalSolver(problem)
solver.solve()

# Objective J is the p-norm of temperature (approximates the peak/maximum temperature)
p_norm = 4
def compute_J_thermal():
    return assemble(T**p_norm * dx)**(1/p_norm)

J_initial = compute_J_thermal()

# =============================================================================
# SCALING ANALYSIS
# =============================================================================
print("--- Starting Scaling Analysis ---")
resolutions = [5, 10, 20, 40, 60, 80, 100, 120, 140, 160]
adj_scaling_times, fd_extrap_times, mesh_nodes = [], [], []

for N in resolutions:
    continue_annotation()
    m_bench = RectangleMesh(N, N, Lx, Ly)
    V_bench = FunctionSpace(m_bench, "CG", 1)
    n_nodes = V_bench.dim()
    mesh_nodes.append(n_nodes)
    
    t_b, T_b, v_b = Function(V_bench).assign(mass_limit), Function(V_bench), TestFunction(V_bench)
    x_b, y_b = SpatialCoordinate(m_bench)
    
    f_s = 250000.0 * exp(-15.0 * ((x_b - Lx/2)**2 + (y_b - Ly/2)**2))
    a_b = k * t_b * inner(grad(TrialFunction(V_bench)), grad(v_b)) * dx
    L_b = f_s * v_b * dx
    bc_b = DirichletBC(V_bench, Constant(20.0), "on_boundary")
    
    # 1. Adjoint Measure
    solve(a_b == L_b, T_b, bcs=bc_b)
    J_b = assemble(T_b**p_norm * dx)**(1/p_norm)
    
    start = time.time()
    dJdt_b = compute_gradient(J_b, Control(t_b))
    _ = assemble(dJdt_b * v_b * dx).dat.data_ro
    adj_scaling_times.append(time.time() - start)
    
    # 2. FD Measure (Sample 10 to extrapolate)
    pause_annotation()
    start = time.time()
    for i in range(10):
        t_b.dat.data[i] += 1e-5
        solve(a_b == L_b, T_b, bcs=bc_b)
        _ = assemble(T_b**p_norm * dx)**(1/p_norm)
        t_b.dat.data[i] -= 1e-5
    fd_extrap_times.append(((time.time() - start) / 10) * n_nodes)
    
    continue_annotation()
    get_working_tape().clear_tape()

plt.figure(figsize=(8, 4))
plt.loglog(mesh_nodes[1:], fd_extrap_times[1:], 'o-', label='Finite Difference (Extrapolated)', color='red')
plt.loglog(mesh_nodes[1:], adj_scaling_times[1:], 's-', label='Adjoint Method', color='blue')
plt.xlabel("Number of Nodes (Mesh Size)"); plt.ylabel("Time [seconds]")
plt.title("Computational Cost Scaling: FD vs Adjoint Gradient (Log-Log)")
plt.legend(); plt.grid(True, which="both", ls="-", alpha=0.5)
plt.show()

## 4 Adjoint-driven optimisation

In [ ]:
# Pre-calculate the constant volume gradient (Integral of basis functions)
vol_grad_constant = assemble(Constant(1.0) * v_test * dx).dat.data_ro[:].copy()
control_adj = Control(rho)

# MMA Parameters
n, m = V.dim(), 1
xval = rho.dat.data[:].reshape((n, 1))
xold1, xold2 = xval.copy(), xval.copy()
xmin, xmax = 0.0 * np.ones((n, 1)), 1.0 * np.ones((n, 1))
low, upp = np.zeros((n, 1)), np.zeros((n, 1))

a0, epsimin = 1.0, 1e-7
a, c, d = np.zeros((m, 1)), 10000.0 * np.ones((m, 1)), np.zeros((m, 1))
raa0, raa0eps = 0.01, 1e-6
raa, raaeps = np.ones((m, 1)) * 0.01, np.ones((m, 1)) * 1e-6
obj_scale = 1.0 / J_initial

continue_annotation()
get_working_tape().clear_tape()

history_rho = []
history_T = []
history_obj = []

n_steps = 50

pbar = tqdm(range(1, n_steps), desc="Adjoint Optimization")
for i in pbar:
    continue_annotation()
    solver.solve()
    
    # Capture fields for animation
    f0val_raw = compute_J_thermal()
    history_rho.append(rho.dat.data.copy())
    history_T.append(T.dat.data.copy())
    history_obj.append(float(f0val_raw))

    f0val_raw = compute_J_thermal()
    current_vol = assemble(t_phys * dx)
    
    f0val = np.array([[float(f0val_raw * obj_scale)]]) 
    df0dx_func = compute_gradient(f0val_raw, control_adj)
    df0dx = (assemble(df0dx_func * v_test * dx).dat.data_ro[:].reshape((n, 1)) * obj_scale)
    
    fval = np.array([[(current_vol / mass_limit) - 1.0]])
    dfdx = (vol_grad_constant * (t_max_adj - t_min_adj) / mass_limit).reshape((m, n))

    pause_annotation()
    get_working_tape().clear_tape()
    
    low, upp, raa0, raa = asymp(i, n, xval, xold1, xold2, xmin, xmax, low, upp, raa0, raa, raa0eps, raaeps, df0dx, dfdx, asyinit=0.1)
    xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx, fval, dfdx, a0, a, c, d)
    
    innerit = 0
    while innerit < 5:
        rho.dat.data[:] = xmma.flatten()
        solver.solve()
        
        f0valnew_raw = compute_J_thermal()
        f0valnew = np.array([[float(f0valnew_raw * obj_scale)]])
        fvalnew = np.array([[(assemble(t_phys * dx) / mass_limit) - 1.0]])

        if concheck(m, epsimin, f0app, f0valnew, fapp, fvalnew):
            break
        innerit += 1
        raa0, raa = raaupdate(xmma, xval, xmin, xmax, low, upp, f0valnew, fvalnew, f0app, fapp, raa0, raa, raa0eps, raaeps, epsimin)
        xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx, fval, dfdx, a0, a, c, d)

    xold2, xold1, xval = xold1.copy(), xval.copy(), xmma.copy()
    rho.dat.data[:] = xval.flatten()
    
    pbar.set_postfix({'Obj': f"{float(f0val_raw):.4f}", 'Vol': f"{float(current_vol):.4f}"})

# Ensure final state is solved
solver.solve()

t_viz = Function(V, name="PlottingThickness")
fig_anim, (ax_rho, ax_T, ax_obj) = plt.subplots(1, 3, figsize=(18, 5))

ax_rho.set_title("Optimised Thickness Evolution")
ax_rho.set_aspect('equal')
ax_T.set_title("Temperature Evolution")
ax_T.set_aspect('equal')

# Setup the objective plot axis
ax_obj.set_title("Objective Convergence")
ax_obj.set_xlabel("Iteration")
ax_obj.set_ylabel("Thermal Compliance")
ax_obj.set_xlim(0, len(history_obj))
ax_obj.set_ylim(min(history_obj) * 0.9, max(history_obj) * 1.1)
ax_obj.grid(True, linestyle='--', alpha=0.6)

frames = []

for idx, (r_data, t_data) in enumerate(zip(history_rho, history_T)):
    rho.dat.data[:] = r_data
    T.dat.data[:] = t_data
    t_viz.interpolate(t_phys)
    
    # 1. Thickness Plot
    c1 = tripcolor(t_viz, axes=ax_rho, cmap='viridis', vmin=t_min_adj, vmax=t_max_adj)
    
    # 2. Temperature Plot
    c2 = tripcolor(T, axes=ax_T, cmap='inferno')
    
    # 3. Objective Line Plot
    line, = ax_obj.plot(range(idx + 1), history_obj[:idx + 1], color='blue', lw=2)
    
    frames.append([c1, c2, line])

plt.tight_layout()
plt.close(fig_anim)
ani = ArtistAnimation(fig_anim, frames, interval=100, blit=True)
display(HTML(f'<div style=\"width:100%;\">{ani.to_jshtml()}</div>'))

## 5. Summary

In this tutorial, we bridged the gap between raw gradient calculation and automated optimisation:
* **Finite Differences** scale poorly ($O(N)$ solves per iteration), making them unviable for realistic topologies.
* **The Adjoint Method** provides exact, analytical gradients in essentially one extra PDE solve ($O(1)$ time), unlocking the potential for spatial topology optimisation.
* **GCMMA** effectively manages complex optimisation problems using these adjoint gradients, smoothly navigating the design space to minimise our peak temperature while respecting the mass limit.

## 6. Try It Yourself

**Alter the Target Mass Fraction**
Change `mass_limit` in the adjoint loop to a smaller value like **0.05** or a larger one like **0.3**.
* *Prediction:* The optimiser will distribute material differently to accommodate the new target mass.

**Modify the p-norm**
The objective uses `p_norm = 4` to approximate the maximum temperature. Try changing this to `p_norm = 1` (which minimises *average* temperature instead of *peak* temperature).
* *Prediction:* The resulting thickness map will be less concentrated around the explicit peak and more smoothly distributed across the domain.

In [ ]:
# ==========================================
# Your Sandbox Workspace
# ==========================================

# Copy the relevant parts of the code from the sections above 
# and modify them here!

# Write your modified solver and plotting code below:
